In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# class Encoder(nn.Module):
#     def __init__(self, input_size, hidden_size):
#         super(Encoder, self).__init__()
#         self.hidden_size = hidden_size
#         self.embedding = nn.Embedding(input_size, hidden_size)
#         self.gru = nn.GRU(hidden_size, hidden_size)

#     def forward(self, input):
#         embedded = self.embedding(input).view(1, 1, -1)
#         output, hidden = self.gru(embedded)
#         return output, hidden

# input_size = 10
# hidden_size = 20
# encoder = Encoder(input_size, hidden_size)
# input_tensor = torch.tensor([5])  # Example input
# output, hidden = encoder(input_tensor)
# print(f"Encoder output shape: {output.shape}")
# print(f"Encoder hidden state shape: {hidden.shape}")

Encoder output shape: torch.Size([1, 1, 20])
Encoder hidden state shape: torch.Size([1, 1, 20])


In [31]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, num_layers, batch_first=True)

    def forward(self, input):
        embedded = self.embedding(input)
        output, hidden = self.gru(embedded)
        return output, hidden

class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden):
        # input: (batch_size,)
        embedded = self.embedding(input).unsqueeze(1)  # (batch_size, 1, hidden_size)
        embedded = F.relu(embedded)
        output, hidden = self.gru(embedded, hidden)  # output: (batch_size, 1, hidden_size)
        output = self.out(output.squeeze(1))  # (batch_size, output_size)
        return output, hidden

    # def forward(self, input, hidden):
    #     output = self.embedding(input).view(1, 1, -1)
    #     output = F.relu(output)
    #     output, hidden = self.gru(output, hidden)
    #     output = self.out(output[0])
    #     return output, hidden

class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        max_len = tgt.size(1)
        vocab_size = self.decoder.output_size

        outputs = torch.zeros(batch_size, max_len, vocab_size).to(src.device)

        encoder_output, hidden = self.encoder(src)

        decoder_input = tgt[:, 0]

        for t in range(1, max_len):
            output, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            decoder_input = tgt[:, t] if teacher_force else output.argmax(1)

        return outputs

In [32]:
vocab_size = 10000
input_vocab_size = vocab_size
output_vocab_size = vocab_size
hidden_size = 256
batch_size = 32
src_len = 10
tgt_len = 10

encoder = EncoderRNN(input_vocab_size, hidden_size)
decoder = DecoderRNN(hidden_size, output_vocab_size)
model = EncoderDecoder(encoder, decoder)

src_seq = torch.randint(0, input_vocab_size, (batch_size, src_len))
tgt_seq = torch.randint(0, output_vocab_size, (batch_size, tgt_len))
output = model(src_seq, tgt_seq)
print(f"Output shape: {output.shape}")

Output shape: torch.Size([32, 10, 10000])


In [34]:
num_epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

# Training loop
for epoch in range(num_epochs):
    output = model(src_seq, tgt_seq)  # (batch, tgt_len, vocab)
    loss = 0
    for t in range(1, tgt_len):
        loss += criterion(output[:, t, :], tgt_seq[:, t])  # per timestep
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

In [39]:
def token_accuracy(predictions, targets, pad_token=None):
    """
    predictions: (batch_size, seq_len)
    targets:     (batch_size, seq_len)
    """
    assert predictions.shape == targets.shape

    if pad_token is not None:
        mask = (targets != pad_token)
        correct = (predictions == targets) & mask
        total = mask.sum()
    else:
        correct = (predictions == targets)
        total = torch.numel(targets)

    accuracy = float(correct.sum()) / total
    return accuracy
token_accuracy(src_seq, tgt_seq)

0.0

RuntimeError: input.size(-1) must be equal to input_size. Expected 256, got 8192